# AE2 – Obtención y reconocimiento del dataset
**Proyecto:** Interacción en publicaciones de TikTok (dataset *TikTok Videos 5.6B*)
**Integrantes / Grupo / Docente:** _completar_

Flujo: (1) obtener una **muestra** del dataset desde Hugging Face y cargarla en un DataFrame, (2) descripción básica, (3) generar `resultados/resumen_dataframe.txt` (UTF-8).

> Ejecutar desde la carpeta `AE2/`. El dataset completo pesa ~460 GB, por eso **no se descarga completo**: se lee una muestra.

## 1. Ficha del dataset
| Campo | Valor |
|---|---|
| Nombre | TikTok Videos (5.6 billion) |
| Identificador | `datasocial/tiktok-5.6B-videos` |
| Enlace | https://huggingface.co/datasets/datasocial/tiktok-5.6B-videos |
| Formato / tamaño | Parquet · 5.598.141.717 filas · 460 GB (según la ficha) |
| Split | `train` (único) · subset `default` |
| Licencia | CC BY-NC 4.0 (uso no comercial) |
| ¿Sintético? | No se declara como sintético |
| Muestra | **Sí**: ver sección 2 |

**Consideración ética:** la ficha indica que los datos se recolectaron con la API móvil de TikTok. Por privacidad **no se cargan** las columnas `caption`, `mentions`, `on_screen_text` ni `hashtag_ids`, y no se publican textos ni identificadores de creadores.

In [ ]:
# %pip install pandas pyarrow numpy huggingface_hub datasets
from datetime import date, datetime
from pathlib import Path
import random
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

In [ ]:
# ---- Parámetros ----
DATASET_ID = "datasocial/tiktok-5.6B-videos"
CONFIG, SPLIT = "default", "train"
ENLACE = f"https://huggingface.co/datasets/{DATASET_ID}"
FECHA_CONSULTA = date.today().isoformat()
TOTAL_FILAS_DATASET = 5_598_141_717      # según la ficha

SEED = 42            # reproducibilidad de la selección del archivo
N_FILAS = 200_000    # tamaño aprox. de la muestra (ajustable a su equipo)
K_GRUPOS = 5         # grupos de filas (row groups) repartidos en el archivo

# Columnas cargadas (se omiten las que contienen texto libre o ids de creadores)
COLUMNAS = ["video_id","posted_at","country","language","type","duration_ms","image_count",
            "width","height","hashtags","sound_id","is_ad","branded_content","product_id",
            "is_shop_video","seller_id","is_ai_generated","not_recommended_for_fyp","sound_muted",
            "views","likes","comments","shares","saves","downloads","engagement_rate","stats_updated_at"]

RUTA_RESUMEN = Path("resultados") / "resumen_dataframe.txt"

In [ ]:
def muestrear_parquet(archivo, columnas, k_grupos, n_total):
    """Lee k grupos de filas repartidos uniformemente en un archivo parquet
    y toma como máximo n_total/k filas de cada uno."""
    pf = pq.ParquetFile(archivo)
    ng = pf.metadata.num_row_groups
    idx = sorted(set(np.linspace(0, ng - 1, min(k_grupos, ng)).astype(int).tolist()))
    por_grupo = max(1, n_total // len(idx))
    tablas = [pf.read_row_group(i, columns=columnas).slice(0, por_grupo) for i in idx]
    return pa.concat_tables(tablas).to_pandas(), ng, idx

In [ ]:
# ---- Obtención de la muestra desde Hugging Face ----
try:
    from huggingface_hub import HfApi, HfFileSystem
    api = HfApi()
    REVISION = api.dataset_info(DATASET_ID).sha
    archivos = sorted(f for f in api.list_repo_files(DATASET_ID, repo_type="dataset") if f.endswith(".parquet"))
    ARCHIVO = random.Random(SEED).choice(archivos)
    with HfFileSystem().open(f"datasets/{DATASET_ID}/{ARCHIVO}", "rb") as f:
        df, n_grupos, idx = muestrear_parquet(f, COLUMNAS, K_GRUPOS, N_FILAS)
    METODO = "parquet por grupos de filas (HfFileSystem + pyarrow)"
    NOTA_MUESTRA = (f"Archivo '{ARCHIVO}' elegido al azar (semilla {SEED}) entre {len(archivos)} archivos parquet; "
                    f"{len(idx)} grupos de filas repartidos de {n_grupos}; hasta {N_FILAS:,} filas. "
                    f"No es una muestra aleatoria simple del dataset completo.")
except Exception as e:
    print("Lectura por parquet falló, se usa streaming (primeras filas):", e)
    from datasets import load_dataset
    ds = load_dataset(DATASET_ID, split=SPLIT, streaming=True).select_columns(COLUMNAS)
    df = pd.DataFrame(list(ds.take(N_FILAS)))
    REVISION, ARCHIVO = "no disponible", "streaming"
    METODO = "datasets streaming (take)"
    NOTA_MUESTRA = (f"Primeras {N_FILAS:,} filas por streaming. Están ordenadas por video_id, por lo que "
                    f"corresponden a los videos más antiguos (2014): muestra muy sesgada en el tiempo.")

ES_MUESTRA = True
print("Método:", METODO); print("Revisión:", REVISION); print("Fecha de consulta:", FECHA_CONSULTA)
print("Muestra:", NOTA_MUESTRA)

## 2. Descripción básica del DataFrame

### 2.1 Algunos registros

In [ ]:
df.head(10)

### 2.2 Dimensiones

In [ ]:
print(f"Filas: {df.shape[0]:,} | Columnas: {df.shape[1]}")
print(f"Fracción del dataset completo: {df.shape[0]/TOTAL_FILAS_DATASET:.6%}")

### 2.3 Nombres y tipos de columnas

In [ ]:
pd.DataFrame({"columna": df.columns, "tipo": df.dtypes.astype(str).values})

### 2.4 Valores nulos por columna

In [ ]:
df.isna().sum().to_frame('nulos')

Según la ficha, hay nulos con significado propio (`is_ai_generated` y `not_recommended_for_fyp`: *null = TikTok no lo indicó*) y valores `0` que significan "ninguno" (`sound_id`, `product_id`, `seller_id`). Además, `country` y `branded_content` pueden venir como texto vacío. Se cuentan aparte.

In [ ]:
def contar_vacios(df):
    """Cadenas vacías o listas vacías en columnas no numéricas."""
    def vacio(v):
        if isinstance(v, str): return v.strip() == ""
        return hasattr(v, "__len__") and len(v) == 0
    out = {}
    for c in df.columns:
        if df[c].dtype == object or str(df[c].dtype) in ("string", "str"):
            out[c] = int(df[c].map(vacio).sum())
    return out
pd.Series(contar_vacios(df), name="vacíos").loc[lambda s: s > 0].to_frame()

### 2.5 Vistazo rápido al contenido

In [ ]:
print("Periodo de posted_at:", df["posted_at"].min(), "→", df["posted_at"].max())
display(df["type"].value_counts(dropna=False).to_frame("publicaciones"))
display(df["country"].replace("", "(vacío)").value_counts().head(10).to_frame("publicaciones"))
df[["duration_ms","views","likes","comments","shares","saves","engagement_rate"]].describe()

## 3. ¿Qué representa cada fila y qué variables respaldan la necesidad?
- **Cada fila** es una publicación de TikTok (video o foto) con una *fotografía* de sus métricas en `stats_updated_at`.
- **Resultado a describir:** `views` y `engagement_rate` (según la ficha, (likes+comments+shares+saves)/views).
- **Variables para comparar grupos:** `type`, `duration_ms` (por rangos), `country`, `language`, `is_ad`, `is_shop_video`, `branded_content`, `is_ai_generated`, y `posted_at` (año).
- Las métricas están muy sesgadas (pocas publicaciones con millones de vistas), por lo que se usarán **medianas** además de promedios.

**Pregunta de análisis (propuesta):** ¿cómo varían las vistas medianas y la tasa de interacción según el tipo de publicación, el rango de duración, el país y si es publicidad o contenido de TikTok Shop?

## 4. Generación automática del resumen (.txt)

In [ ]:
def generar_resumen(df, ruta, dataset_id, split, config=None, revision=None, enlace=None,
                    fecha_consulta=None, es_muestra=False, nota_muestra="", total_filas=None):
    """Escribe el resumen del DataFrame en UTF-8; crea la carpeta y sobrescribe en cada ejecución."""
    ruta = Path(ruta); ruta.parent.mkdir(parents=True, exist_ok=True)
    nulos = df.isna().sum(); vacios = contar_vacios(df)
    L = ["RESUMEN DEL DATAFRAME", "=" * 60,
         f"Generado el: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}", "",
         "1. ORIGEN DE LOS DATOS", f"- Identificador del dataset: {dataset_id}"]
    if enlace: L.append(f"- Enlace: {enlace}")
    L.append(f"- Split utilizado: {split}")
    L.append(f"- Configuración (subset): {config if config else 'no aplica / por defecto'}")
    if revision: L.append(f"- Revisión (commit): {revision}")
    if fecha_consulta: L.append(f"- Fecha de consulta: {fecha_consulta}")
    if total_filas: L.append(f"- Filas del dataset completo (según la ficha): {total_filas:,}")
    L.append("- ¿Muestra?: " + (f"SI. {nota_muestra}" if es_muestra else "No, DataFrame completo del split"))
    L += ["", "2. DIMENSIONES", f"- Filas: {df.shape[0]}", f"- Columnas: {df.shape[1]}", "",
          "3. COLUMNAS, TIPOS Y VALORES NULOS"]
    ancho = max(len(str(c)) for c in df.columns)
    L.append(f"{'Columna'.ljust(ancho)} | {'Tipo'.ljust(18)} | Nulos")
    L.append("-" * (ancho + 32))
    for c in df.columns:
        L.append(f"{str(c).ljust(ancho)} | {str(df[c].dtype).ljust(18)} | {int(nulos[c])}")
    L += ["", f"Total de valores nulos (isna): {int(nulos.sum())}", "",
          "4. NOTA: VALORES VACIOS (cadena vacia o lista vacia)"]
    con = {c: n for c, n in vacios.items() if n > 0}
    L += [f"- {c}: {n}" for c, n in con.items()] if con else ["- No se encontraron valores vacios."]
    if "posted_at" in df.columns and df["posted_at"].notna().any():
        L += ["", "5. PERIODO CUBIERTO POR LA MUESTRA (posted_at)",
              f"- Minimo: {df['posted_at'].min()}", f"- Maximo: {df['posted_at'].max()}"]
    L.append("")
    ruta.write_text("\n".join(L), encoding="utf-8")
    return ruta

In [ ]:
ruta = generar_resumen(df, RUTA_RESUMEN, DATASET_ID, SPLIT, config=CONFIG, revision=REVISION,
    enlace=ENLACE, fecha_consulta=FECHA_CONSULTA, es_muestra=ES_MUESTRA,
    nota_muestra=NOTA_MUESTRA, total_filas=TOTAL_FILAS_DATASET)
print("Archivo generado en:", ruta.resolve())

### Verificación del contenido del archivo generado

In [ ]:
print(RUTA_RESUMEN.read_text(encoding="utf-8"))

## 5. Limitaciones a tener presentes
- Se analiza una **muestra** (una fracción mínima del dataset); no representa a TikTok completo.
- Las métricas se midieron en fechas distintas (`stats_updated_at`), así que no son comparables como si fueran del mismo día.
- Videos antiguos vienen de un archivo: algunos campos son 0 o nulos (tamaño, descargas, marca de IA) y la duración está redondeada al segundo (según la ficha).
- `country` y `language` son etiquetas de TikTok (`un` = indeterminado); pueden venir vacías.
- Comparaciones descriptivas: no implican causalidad.